# Baseline setup and CTC feasibility

Historical research source, original zero-based cells **17?23**. Outputs have been cleared.

These cells retain the original global state, Colab paths, manual annotations, and checkpoint assumptions. This notebook is not standalone; earlier state and external artifacts may be required. Some repair branches are alternatives. Read [the migration guide](../../docs/migration.md) before selecting cells to run. For a new baseline, use [the package workflow](../amharic_asr.ipynb).


## Original source cell 17

Position 18 in the original notebook; original code comments may use different numbering.


In [ ]:

import json
import numpy as np
import torch

from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from torchcodec.decoders import AudioDecoder

from transformers import (
    Wav2Vec2Config,
    Wav2Vec2FeatureExtractor,
    Wav2Vec2Processor
)

# ------------------------------------------
# 1. Load the final 25-hour training manifest
# ------------------------------------------

with FINAL_MANIFEST.open(encoding="utf-8") as f:
    train_records = [json.loads(line) for line in f]

assert len(train_records) == 5084

MODEL_ID = "facebook/wav2vec2-xls-r-300m"
SAMPLE_RATE = 16000

model_config = Wav2Vec2Config.from_pretrained(MODEL_ID)

# XLS-R uses layer normalization in its feature extractor,
# so attention masks are appropriate for padded audio.
assert model_config.feat_extract_norm == "layer"

feature_extractor = Wav2Vec2FeatureExtractor(
    feature_size=1,
    sampling_rate=SAMPLE_RATE,
    padding_value=0.0,
    do_normalize=True,
    return_attention_mask=True
)

processor = Wav2Vec2Processor(
    feature_extractor=feature_extractor,
    tokenizer=tokenizer
)

processor.save_pretrained(str(PROJECT_DIR / "processor"))

# ------------------------------------------
# 2. Check CTC sequence-length feasibility
# ------------------------------------------

def output_frames(num_samples):
    length = int(num_samples)

    for kernel, stride in zip(
        model_config.conv_kernel,
        model_config.conv_stride
    ):
        length = (length - kernel) // stride + 1

    return length

ctc_failures = []
minimum_margin = float("inf")
max_label_length = 0

for record in train_records:
    labels = tokenizer.encode(
        record["transcription"],
        add_special_tokens=False
    )

    # CTC needs extra time steps for repeated adjacent labels.
    repeated = sum(
        labels[i] == labels[i - 1]
        for i in range(1, len(labels))
    )

    required_frames = len(labels) + repeated

    # Estimated from measured audio duration.
    estimated_samples = round(
        record["duration_seconds"] * SAMPLE_RATE
    )

    frames = output_frames(estimated_samples)
    margin = frames - required_frames

    minimum_margin = min(minimum_margin, margin)
    max_label_length = max(max_label_length, len(labels))

    if margin < 0:
        ctc_failures.append({
            "id": record["id"],
            "available_frames": frames,
            "required_frames": required_frames
        })

print("CTC FEASIBILITY CHECK")
print("-" * 45)
print("Recordings checked:", len(train_records))
print("Maximum label length:", max_label_length)
print("Minimum estimated frame margin:", minimum_margin)
print("Potentially infeasible recordings:", len(ctc_failures))

if ctc_failures:
    print("First failures:", ctc_failures[:5])

# ------------------------------------------
# 3. Create an on-demand audio dataset
# ------------------------------------------

class AmharicASRDataset(Dataset):
    def __init__(self, records, audio_dir, processor):
        self.records = records
        self.audio_dir = Path(audio_dir)
        self.processor = processor

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        record = self.records[index]

        audio_path = self.audio_dir / record["audio_file"]

        decoder = AudioDecoder(
            str(audio_path),
            sample_rate=SAMPLE_RATE,
            num_channels=1
        )

        audio_samples = decoder.get_all_samples()

        waveform = (
            audio_samples.data[0]
            .cpu()
            .numpy()
            .astype(np.float32)
        )

        assert audio_samples.sample_rate == SAMPLE_RATE

        inputs = self.processor.feature_extractor(
            waveform,
            sampling_rate=SAMPLE_RATE
        )

        labels = self.processor.tokenizer(
            record["transcription"],
            add_special_tokens=False
        ).input_ids

        return {
            "input_values": inputs.input_values[0],
            "labels": labels,
            "id": record["id"]
        }

# ------------------------------------------
# 4. Dynamically pad each training batch
# ------------------------------------------

def ctc_collator(examples):
    audio_features = [
        {"input_values": x["input_values"]}
        for x in examples
    ]

    label_features = [
        {"input_ids": x["labels"]}
        for x in examples
    ]

    batch = processor.feature_extractor.pad(
        audio_features,
        padding=True,
        return_attention_mask=True,
        return_tensors="pt"
    )

    label_batch = processor.tokenizer.pad(
        label_features,
        padding=True,
        return_tensors="pt"
    )

    batch["labels"] = label_batch["input_ids"].masked_fill(
        label_batch["attention_mask"] != 1,
        -100
    )

    return batch

train_dataset = AmharicASRDataset(
    train_records,
    AUDIO_DIR,
    processor
)

# Small diagnostic batch, CPU only.
test_loader = DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=False,
    collate_fn=ctc_collator,
    num_workers=0
)

batch = next(iter(test_loader))

# ------------------------------------------
# 5. Verify the resulting tensors
# ------------------------------------------

print("\nDATALOADER TEST")
print("-" * 45)

print("Dataset size:", len(train_dataset))
print("Audio tensor shape:", tuple(batch["input_values"].shape))
print("Attention mask shape:", tuple(batch["attention_mask"].shape))
print("Label tensor shape:", tuple(batch["labels"].shape))

print("Audio dtype:", batch["input_values"].dtype)
print("Label dtype:", batch["labels"].dtype)

print(
    "Audio contains NaN/Inf:",
    not torch.isfinite(batch["input_values"]).all().item()
)

print(
    "Valid label counts:",
    (batch["labels"] != -100).sum(dim=1).tolist()
)

print(
    "Audio lengths (seconds):",
    [
        round(int(x) / SAMPLE_RATE, 2)
        for x in batch["attention_mask"].sum(dim=1)
    ]
)

assert torch.isfinite(batch["input_values"]).all()
assert not ctc_failures, "Some recordings fail estimated CTC feasibility."

print("\nDATALOADER TEST PASSED")


## Original source cell 18

Position 19 in the original notebook; original code comments may use different numbering.


In [ ]:

import json
import numpy as np

from IPython.display import Audio, display
from torchcodec.decoders import AudioDecoder
from torch.utils.data import DataLoader

# -------------------------------------------------
# 1. Calculate minimum frames required by CTC
# -------------------------------------------------

def required_ctc_frames(text):
    ids = tokenizer.encode(
        text,
        add_special_tokens=False
    )

    repeated = sum(
        ids[i] == ids[i - 1]
        for i in range(1, len(ids))
    )

    return len(ids) + repeated


# -------------------------------------------------
# 2. Verify actual audio lengths near CTC limits
# -------------------------------------------------

valid_records = []
excluded_records = []
inspected = []

for record in train_records:
    required = required_ctc_frames(record["transcription"])

    estimated_samples = round(
        record["duration_seconds"] * SAMPLE_RATE
    )
    estimated_frames = output_frames(estimated_samples)

    # Metadata is approximate. Decode recordings close
    # to the minimum instead of relying on estimates.
    if estimated_frames - required <= 10:
        path = AUDIO_DIR / record["audio_file"]

        decoder = AudioDecoder(
            str(path),
            sample_rate=SAMPLE_RATE,
            num_channels=1
        )

        samples = decoder.get_all_samples()
        actual_frames = output_frames(samples.data.shape[-1])

        inspected.append(record["id"])

        if actual_frames < required:
            excluded_records.append({
                **record,
                "required_frames": required,
                "available_frames": actual_frames
            })

            print("\nINFEASIBLE RECORDING")
            print("ID:", record["id"])
            print(
                "Actual duration:",
                round(samples.data.shape[-1] / SAMPLE_RATE, 3),
                "seconds"
            )
            print("Available frames:", actual_frames)
            print("Required frames:", required)
            print("Transcription:")
            print(record["transcription"])

            print("\nAudio playback:")
            display(
                Audio(
                    samples.data[0].cpu().numpy(),
                    rate=SAMPLE_RATE
                )
            )
            continue

    valid_records.append(record)


# -------------------------------------------------
# 3. Save a new, corrected training manifest
# -------------------------------------------------

SAFE_MANIFEST = (
    DATA_DIR / "train_final_25h_ctc_safe.jsonl"
)

with SAFE_MANIFEST.open("w", encoding="utf-8") as file:
    for record in valid_records:
        file.write(
            json.dumps(record, ensure_ascii=False) + "\n"
        )

# Keep an audit record of exclusions.
EXCLUSIONS_PATH = (
    DATA_DIR / "ctc_exclusions_25h.json"
)

with EXCLUSIONS_PATH.open("w", encoding="utf-8") as file:
    json.dump(
        excluded_records,
        file,
        ensure_ascii=False,
        indent=2
    )


# -------------------------------------------------
# 4. Rebuild the DataLoader with valid recordings
# -------------------------------------------------

train_records = valid_records

train_dataset = AmharicASRDataset(
    train_records,
    AUDIO_DIR,
    processor
)

test_loader = DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=False,
    collate_fn=ctc_collator,
    num_workers=0
)

batch = next(iter(test_loader))

assert torch.isfinite(batch["input_values"]).all()
assert len(valid_records) + len(excluded_records) == 5084

total_hours = sum(
    r["duration_seconds"] for r in valid_records
) / 3600

print("\nFINAL DATASET VALIDATION")
print("-" * 45)
print("Original recordings:", 5084)
print("Near-limit recordings decoded:", len(inspected))
print("Excluded recordings:", len(excluded_records))
print("Final recordings:", len(train_records))
print("Final training hours:", round(total_hours, 4))
print(
    "Unique speakers:",
    len({r["speaker_id"] for r in train_records})
)
print("Batch audio shape:", tuple(batch["input_values"].shape))
print("Batch label shape:", tuple(batch["labels"].shape))
print("Safe manifest:", SAFE_MANIFEST)

print("\nDATALOADER VALIDATION PASSED")


## Original source cell 19

Position 20 in the original notebook; original code comments may use different numbering.


In [ ]:

import gc
import torch

from transformers import Wav2Vec2ForCTC

# ------------------------------------------
# 1. Load pretrained XLS-R
# ------------------------------------------

DEVICE = "cuda"
UNFROZEN_LAYERS = 2

assert torch.cuda.is_available(), "GPU is required."

model = Wav2Vec2ForCTC.from_pretrained(
    MODEL_ID,
    vocab_size=len(tokenizer),
    pad_token_id=tokenizer.pad_token_id,
    ctc_loss_reduction="mean",
    ctc_zero_infinity=True,
    ignore_mismatched_sizes=True
)

# A newly initialized CTC head is expected.
assert model.lm_head.out_features == 321

# ------------------------------------------
# 2. Freeze everything, then selectively
#    unfreeze the layers we want to train
# ------------------------------------------

for parameter in model.parameters():
    parameter.requires_grad = False

for layer in model.wav2vec2.encoder.layers[-UNFROZEN_LAYERS:]:
    for parameter in layer.parameters():
        parameter.requires_grad = True

for parameter in model.lm_head.parameters():
    parameter.requires_grad = True

# Avoid the frozen-input checkpointing problem.
model.gradient_checkpointing_enable(
    gradient_checkpointing_kwargs={
        "use_reentrant": False
    }
)

model.to(DEVICE)
model.train()

total_params = sum(
    p.numel() for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("MODEL CONFIGURATION")
print("-" * 45)
print("Total parameters:", f"{total_params:,}")
print("Trainable parameters:", f"{trainable_params:,}")
print(
    "Trainable percentage:",
    f"{100 * trainable_params / total_params:.2f}%"
)
print("Transformer layers:", len(model.wav2vec2.encoder.layers))
print("Unfrozen layers:", UNFROZEN_LAYERS)
print("Vocabulary size:", model.config.vocab_size)
print("CTC blank ID:", model.config.pad_token_id)
print(
    "Gradient checkpointing:",
    model.is_gradient_checkpointing
)

# ------------------------------------------
# 3. Choose a short valid recording
# ------------------------------------------

sample_index = min(
    range(min(64, len(train_records))),
    key=lambda i: train_records[i]["duration_seconds"]
)

sample_record = train_records[sample_index]

# Reuse the verified dataset and collator.
sample_batch = ctc_collator(
    [train_dataset[sample_index]]
)

sample_batch = {
    key: value.to(DEVICE)
    for key, value in sample_batch.items()
}

print("\nDIAGNOSTIC RECORDING")
print("ID:", sample_record["id"])
print(
    "Duration:",
    round(sample_record["duration_seconds"], 2),
    "seconds"
)
print(
    "Label tokens:",
    int((sample_batch["labels"] != -100).sum())
)

# ------------------------------------------
# 4. Forward pass + backward pass
# ------------------------------------------

torch.cuda.reset_peak_memory_stats()
model.zero_grad(set_to_none=True)

scaler = torch.amp.GradScaler(
    "cuda",
    init_scale=128.0
)

with torch.autocast(
    device_type="cuda",
    dtype=torch.float16
):
    outputs = model(**sample_batch)
    loss = outputs.loss

print("\nFORWARD PASS")
print("CTC loss:", round(float(loss.item()), 4))
print("Finite loss:", bool(torch.isfinite(loss).item()))
print("Output logits:", tuple(outputs.logits.shape))

assert torch.isfinite(loss), "Non-finite CTC loss."

# Scale gradients to make FP16 training more stable.
scaler.scale(loss).backward()

# ------------------------------------------
# 5. Verify gradients in every trainable
#    component
# ------------------------------------------

def gradient_report(module, name):
    gradients = [
        p.grad
        for p in module.parameters()
        if p.requires_grad
    ]

    available = [
        g for g in gradients
        if g is not None
    ]

    all_present = len(available) == len(gradients)

    all_finite = (
        len(available) > 0
        and all(torch.isfinite(g).all().item() for g in available)
    )

    # Gradients are currently scaled by GradScaler.
    scale = scaler.get_scale()

    max_gradient = max(
        (g.detach().abs().max().item() / scale
         for g in available),
        default=0.0
    )

    print(
        f"{name:20} "
        f"gradients={len(available)}/{len(gradients)} "
        f"finite={all_finite} "
        f"max_abs={max_gradient:.8f}"
    )

    assert all_present, f"Missing gradients in {name}"
    assert all_finite, f"Non-finite gradients in {name}"
    assert max_gradient > 0, f"Zero gradients in {name}"


print("\nGRADIENT FLOW CHECK")
print("-" * 70)

gradient_report(
    model.wav2vec2.encoder.layers[-2],
    "Transformer layer 22"
)

gradient_report(
    model.wav2vec2.encoder.layers[-1],
    "Transformer layer 23"
)

gradient_report(
    model.lm_head,
    "CTC prediction head"
)

print(
    "\nPeak GPU memory:",
    round(torch.cuda.max_memory_allocated() / 1024**3, 2),
    "GB"
)

# No optimizer.step() here.
# This is a diagnostic, not a training update.
model.zero_grad(set_to_none=True)

del outputs, loss, sample_batch
gc.collect()
torch.cuda.empty_cache()

print("\nGRADIENT DIAGNOSTIC PASSED")
print("No model weights were updated.")


## Original source cell 20

Position 21 in the original notebook; original code comments may use different numbering.


In [ ]:

# CELL 21 - RESTORE EXISTING BASELINE
# Do not recreate or overwrite saved checkpoints.

import json
import shutil
from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Amharic_ASR_XLSR"
)
DATA_DIR = PROJECT_DIR / "data"
CHECKPOINT_DIR = PROJECT_DIR / "checkpoints"

BASELINE_DIR = (
    PROJECT_DIR / "baselines" / "xlsr_step0"
)

BASELINE_EVAL_PATH = (
    DATA_DIR / "baseline_eval_fixed_12.jsonl"
)

DRIVE_EVAL_AUDIO = PROJECT_DIR / "evaluation_audio"
EVAL_AUDIO_DIR = Path("/content/amharic_eval_audio")
EVAL_AUDIO_DIR.mkdir(parents=True, exist_ok=True)

# 1. Verify the original checkpoint.
assert BASELINE_DIR.is_dir(), (
    f"Missing baseline: {BASELINE_DIR}"
)

assert list(BASELINE_DIR.glob("*.safetensors")), (
    "Baseline model weights are missing."
)

print("Original baseline checkpoint: FOUND")

# 2. Restore the fixed evaluation manifest.
assert BASELINE_EVAL_PATH.is_file(), (
    f"Missing evaluation manifest: {BASELINE_EVAL_PATH}"
)

with BASELINE_EVAL_PATH.open(encoding="utf-8") as f:
    eval_records = [json.loads(line) for line in f]

assert len(eval_records) == 12
assert len({r["id"] for r in eval_records}) == 12

print("Fixed evaluation recordings:", len(eval_records))

# 3. Restore evaluation audio from Google Drive.
restored = 0
missing_audio = []

for record in eval_records:
    filename = record["id"] + ".mp3"

    source = DRIVE_EVAL_AUDIO / filename
    destination = EVAL_AUDIO_DIR / filename

    if destination.exists() and destination.stat().st_size > 0:
        continue

    if source.exists():
        shutil.copy2(source, destination)
        restored += 1
    else:
        missing_audio.append(filename)

print("Audio files restored locally:", restored)
print("Missing evaluation recordings:", len(missing_audio))

# 4. Locate previously saved baseline results.
BASELINE_METRICS_PATH = (
    DATA_DIR / "baseline_step0_metrics.json"
)

BASELINE_PREDICTIONS_PATH = (
    DATA_DIR / "baseline_step0_predictions_12.jsonl"
)

if BASELINE_METRICS_PATH.exists():
    with BASELINE_METRICS_PATH.open(encoding="utf-8") as f:
        baseline_metrics = json.load(f)

    print(
        "Saved baseline CER:",
        f"{baseline_metrics['cer'] * 100:.2f}%"
    )
    print(
        "Saved baseline WER:",
        f"{baseline_metrics['wer'] * 100:.2f}%"
    )
else:
    print("Baseline metrics not found.")

print("\n" + "=" * 50)
print("BASELINE RECOVERY COMPLETE")
print("=" * 50)
print("Original model preserved:", True)
print("Evaluation manifest:", BASELINE_EVAL_PATH)
print("Evaluation audio directory:", EVAL_AUDIO_DIR)
print("Saved predictions:", BASELINE_PREDICTIONS_PATH.exists())
print("Saved metrics:", BASELINE_METRICS_PATH.exists())

if missing_audio:
    print("\nMissing audio IDs:", missing_audio)
else:
    print("\nAll evaluation audio is available.")


## Original source cell 21

Position 22 in the original notebook; original code comments may use different numbering.


In [ ]:

import json
import shutil
import torch
import numpy as np
import fsspec
import pyarrow.parquet as pq

from pathlib import Path
from tqdm.auto import tqdm
from torchcodec.decoders import AudioDecoder
from jiwer import cer, wer

# =========================================
# 1. Load our fixed evaluation examples
# =========================================

with BASELINE_EVAL_PATH.open(encoding="utf-8") as f:
    eval_records = [json.loads(line) for line in f]

assert len(eval_records) == 12

EVAL_AUDIO_DIR = Path("/content/amharic_eval_audio")
EVAL_AUDIO_DIR.mkdir(parents=True, exist_ok=True)

DRIVE_EVAL_AUDIO = PROJECT_DIR / "evaluation_audio"
DRIVE_EVAL_AUDIO.mkdir(parents=True, exist_ok=True)

needed_ids = {r["id"] for r in eval_records}

# Restore already downloaded audio if available.
for record in eval_records:
    filename = record["id"] + ".mp3"
    local_path = EVAL_AUDIO_DIR / filename
    drive_path = DRIVE_EVAL_AUDIO / filename

    if not local_path.exists() and drive_path.exists():
        shutil.copy2(drive_path, local_path)

missing_ids = {
    record_id for record_id in needed_ids
    if not (EVAL_AUDIO_DIR / (record_id + ".mp3")).exists()
}

print("Evaluation examples:", len(eval_records))
print("Recordings needing download:", len(missing_ids))

# =========================================
# 2. Download only needed validation audio
# =========================================

if missing_ids:
    for shard in tqdm(val_shards, desc="Retrieving validation audio"):
        with fsspec.open(
            shard["url"],
            "rb",
            block_size=8 * 1024**2
        ) as source:

            parquet = pq.ParquetFile(source)

            for group_index in range(parquet.metadata.num_row_groups):

                # First read the lightweight ID column.
                group_ids = parquet.read_row_group(
                    group_index,
                    columns=["id"]
                ).column("id").to_pylist()

                matched = [
                    (index, record_id)
                    for index, record_id in enumerate(group_ids)
                    if record_id in missing_ids
                ]

                if not matched:
                    continue

                # Read audio only for relevant row groups.
                table = parquet.read_row_group(
                    group_index,
                    columns=["audio"]
                )

                for index, record_id in matched:
                    audio = table.column("audio")[index].as_py()
                    encoded = audio.get("bytes")

                    if encoded is None:
                        raise RuntimeError(
                            f"Missing embedded audio: {record_id}"
                        )

                    filename = record_id + ".mp3"
                    local_path = EVAL_AUDIO_DIR / filename
                    drive_path = DRIVE_EVAL_AUDIO / filename

                    local_path.write_bytes(encoded)
                    shutil.copy2(local_path, drive_path)

                    missing_ids.remove(record_id)

        if not missing_ids:
            break

assert not missing_ids, (
    f"Missing evaluation audio IDs: {missing_ids}"
)

print("Evaluation audio ready.")

# =========================================
# 3. Run the STEP-ZERO model
# =========================================

model.eval()

results = []
references = []
predictions = []

for i, record in enumerate(
    tqdm(eval_records, desc="Step-zero inference"), 1
):
    path = EVAL_AUDIO_DIR / (record["id"] + ".mp3")

    decoder = AudioDecoder(
        str(path),
        sample_rate=SAMPLE_RATE,
        num_channels=1
    )

    samples = decoder.get_all_samples()

    waveform = (
        samples.data[0]
        .cpu()
        .numpy()
        .astype(np.float32)
    )

    inputs = processor.feature_extractor(
        waveform,
        sampling_rate=SAMPLE_RATE,
        return_attention_mask=True,
        return_tensors="pt"
    )

    inputs = {
        key: value.to(DEVICE)
        for key, value in inputs.items()
    }

    with torch.inference_mode():
        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):
            logits = model(**inputs).logits

    token_ids = logits.argmax(dim=-1)

    # Standard CTC decoding:
    # collapse repeated frame predictions and remove blanks.
    predicted = processor.batch_decode(
        token_ids.cpu().numpy()
    )[0]

    predicted = normalize_amharic_ctc(predicted)
    reference = record["transcription"]

    blank_fraction = (
        (token_ids == tokenizer.pad_token_id)
        .float()
        .mean()
        .item()
    )

    references.append(reference)
    predictions.append(predicted)

    results.append({
        "id": record["id"],
        "speaker_id": record["speaker_id"],
        "duration_seconds": round(
            len(waveform) / SAMPLE_RATE, 3
        ),
        "reference": reference,
        "prediction": predicted,
        "cer": cer(reference, predicted),
        "wer": wer(reference, predicted),
        "blank_fraction": blank_fraction
    })

    print(f"\n{'=' * 65}")
    print(f"RECORDING {i}")
    print("Duration:", results[-1]["duration_seconds"], "s")
    print("REFERENCE:", reference[:220])
    print("PREDICTED:", predicted[:220])
    print("CER:", round(results[-1]["cer"], 4))
    print("WER:", round(results[-1]["wer"], 4))
    print(
        "Blank-frame percentage:",
        round(blank_fraction * 100, 2),
        "%"
    )

    del inputs, logits, token_ids

# =========================================
# 4. Compute and save baseline metrics
# =========================================

overall_cer = cer(references, predictions)
overall_wer = wer(references, predictions)

STEP0_RESULTS = (
    DATA_DIR / "baseline_step0_predictions_12.jsonl"
)

with STEP0_RESULTS.open("w", encoding="utf-8") as file:
    for result in results:
        file.write(
            json.dumps(result, ensure_ascii=False) + "\n"
        )

STEP0_METRICS = DATA_DIR / "baseline_step0_metrics.json"

metrics = {
    "model": MODEL_ID,
    "checkpoint": str(BASELINE_DIR),
    "training_step": 0,
    "evaluation_examples": len(results),
    "cer": overall_cer,
    "wer": overall_wer,
    "empty_predictions": sum(
        not prediction for prediction in predictions
    ),
    "mean_blank_fraction": float(np.mean([
        r["blank_fraction"] for r in results
    ]))
}

with STEP0_METRICS.open("w", encoding="utf-8") as file:
    json.dump(metrics, file, indent=2)

# Restore training mode for subsequent experiments.
model.train()

print("\n" + "=" * 65)
print("STARTING MODEL BASELINE")
print("=" * 65)

print("Evaluation recordings:", len(results))
print("Overall CER:", round(overall_cer, 4))
print("Overall WER:", round(overall_wer, 4))
print("CER percentage:", round(overall_cer * 100, 2), "%")
print("WER percentage:", round(overall_wer * 100, 2), "%")
print("Empty predictions:", metrics["empty_predictions"])
print(
    "Mean blank-frame percentage:",
    round(metrics["mean_blank_fraction"] * 100, 2),
    "%"
)

print("\nPredictions saved:", STEP0_RESULTS)
print("Metrics saved:", STEP0_METRICS)
print("Evaluation audio:", DRIVE_EVAL_AUDIO)

print("\nSTEP-ZERO BASELINE COMPLETE")


## Original source cell 22

Position 23 in the original notebook; original code comments may use different numbering.


In [ ]:

import random
import math
import gc
import torch

# ==========================================
# 1. CONFIGURATION
# ==========================================

HEAD_LR = 1e-4
ENCODER_LR = 1e-5
WEIGHT_DECAY = 0.01
GRAD_ACCUM_STEPS = 4
MAX_GRAD_NORM = 1.0
SEED = 42

# ==========================================
# 2. SELECT FOUR SUITABLE RECORDINGS
# ==========================================

rng = random.Random(SEED)

eligible = [
    (i, record)
    for i, record in enumerate(train_records)
    if 8 <= record["duration_seconds"] <= 14
    and (
        output_frames(
            round(record["duration_seconds"] * SAMPLE_RATE)
        )
        - required_ctc_frames(record["transcription"])
    ) >= 30
]

rng.shuffle(eligible)

selected_indices = []
selected_speakers = set()

for index, record in eligible:
    if record["speaker_id"] not in selected_speakers:
        selected_indices.append(index)
        selected_speakers.add(record["speaker_id"])

    if len(selected_indices) == GRAD_ACCUM_STEPS:
        break

assert len(selected_indices) == GRAD_ACCUM_STEPS

print("OPTIMIZER DIAGNOSTIC")
print("-" * 50)
print("Selected recordings:", len(selected_indices))
print("Distinct speakers:", len(selected_speakers))
print("Encoder learning rate:", ENCODER_LR)
print("CTC head learning rate:", HEAD_LR)
print("Gradient accumulation:", GRAD_ACCUM_STEPS)

# ==========================================
# 3. CREATE OPTIMIZER
# ==========================================

encoder_parameters = [
    p
    for layer in model.wav2vec2.encoder.layers[-2:]
    for p in layer.parameters()
    if p.requires_grad
]

head_parameters = [
    p for p in model.lm_head.parameters()
    if p.requires_grad
]

optimizer = torch.optim.AdamW(
    [
        {
            "params": encoder_parameters,
            "lr": ENCODER_LR
        },
        {
            "params": head_parameters,
            "lr": HEAD_LR
        }
    ],
    weight_decay=WEIGHT_DECAY
)

scaler = torch.amp.GradScaler(
    "cuda",
    init_scale=128.0
)

# ==========================================
# 4. BACK UP TRAINABLE WEIGHTS IN MEMORY
# ==========================================

original_weights = {
    name: parameter.detach().cpu().clone()
    for name, parameter in model.named_parameters()
    if parameter.requires_grad
}

model.train()
optimizer.zero_grad(set_to_none=True)

losses = []
step_applied = False

try:
    # ======================================
    # 5. ACCUMULATE GRADIENTS
    # ======================================

    for step, index in enumerate(selected_indices, 1):
        batch = ctc_collator([train_dataset[index]])

        batch = {
            key: value.to(DEVICE)
            for key, value in batch.items()
        }

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):
            outputs = model(**batch)
            raw_loss = outputs.loss

        assert torch.isfinite(raw_loss).item(), (
            f"Non-finite loss on recording {step}"
        )

        losses.append(raw_loss.detach().item())

        scaled_loss = raw_loss / GRAD_ACCUM_STEPS
        scaler.scale(scaled_loss).backward()

        print(
            f"Recording {step}/{GRAD_ACCUM_STEPS} "
            f"| CTC loss: {raw_loss.item():.4f}"
        )

        del batch, outputs, raw_loss, scaled_loss

    # ======================================
    # 6. CLIP GRADIENTS AND UPDATE WEIGHTS
    # ======================================

    scaler.unscale_(optimizer)

    trainable_parameters = (
        encoder_parameters + head_parameters
    )

    gradient_norm = torch.nn.utils.clip_grad_norm_(
        trainable_parameters,
        max_norm=MAX_GRAD_NORM,
        error_if_nonfinite=True
    )

    scale_before = scaler.get_scale()

    scaler.step(optimizer)
    scaler.update()

    scale_after = scaler.get_scale()

    # ======================================
    # 7. MEASURE ACTUAL WEIGHT CHANGES
    # ======================================

    changes = {}

    with torch.no_grad():
        for name, parameter in model.named_parameters():
            if name not in original_weights:
                continue

            before = original_weights[name]
            after = parameter.detach().cpu()

            changes[name] = (
                after - before
            ).abs().max().item()

    layer22_change = max(
        value for name, value in changes.items()
        if "encoder.layers.22." in name
    )

    layer23_change = max(
        value for name, value in changes.items()
        if "encoder.layers.23." in name
    )

    head_change = max(
        value for name, value in changes.items()
        if name.startswith("lm_head.")
    )

    step_applied = (
        scale_after >= scale_before
        and layer22_change > 0
        and layer23_change > 0
        and head_change > 0
    )

    print("\nOPTIMIZER RESULTS")
    print("-" * 50)

    print("Mean CTC loss:", round(sum(losses) / len(losses), 4))
    print("Gradient norm before clipping:", float(gradient_norm))
    print("GradScaler before:", scale_before)
    print("GradScaler after:", scale_after)

    print("\nMAXIMUM ABSOLUTE WEIGHT CHANGES")
    print(f"Layer 22: {layer22_change:.10f}")
    print(f"Layer 23: {layer23_change:.10f}")
    print(f"CTC head: {head_change:.10f}")

    print("\nOptimizer update succeeded:", step_applied)

finally:
    # ======================================
    # 8. RESTORE EXACT STARTING WEIGHTS
    # ======================================

    with torch.no_grad():
        for name, parameter in model.named_parameters():
            if name in original_weights:
                parameter.copy_(
                    original_weights[name].to(parameter.device)
                )

    model.zero_grad(set_to_none=True)

    del optimizer, scaler
    gc.collect()
    torch.cuda.empty_cache()

    restored = all(
        torch.equal(
            parameter.detach().cpu(),
            original_weights[name]
        )
        for name, parameter in model.named_parameters()
        if name in original_weights
    )

    print("\nOriginal weights restored:", restored)

assert restored, "Failed to restore original weights."
assert step_applied, "Optimizer failed to update all components."

print("\nOPTIMIZER DIAGNOSTIC PASSED")
print("Model remains at training step zero.")


## Original source cell 23

Position 24 in the original notebook; original code comments may use different numbering.


In [ ]:

# CELL 19B - RESTORE CTC-SAFE TRAINING DATASET

import json
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/Amharic_ASR_XLSR")
DATA_DIR = PROJECT_DIR / "data"
CHECKPOINT_DIR = PROJECT_DIR / "checkpoints"

SAFE_MANIFEST = DATA_DIR / "train_final_25h_ctc_safe.jsonl"

# Verify the previously saved corrected dataset.
assert SAFE_MANIFEST.exists(), (
    f"Corrected dataset not found: {SAFE_MANIFEST}"
)

with SAFE_MANIFEST.open(encoding="utf-8") as f:
    safe_records = [json.loads(line) for line in f]

assert len(safe_records) == 5083, (
    f"Expected 5083 recordings, found {len(safe_records)}"
)

# Ensure the invalid recording is excluded.
invalid_id = "5psPJBBetW3jsIWVgElP"

assert all(
    record["id"] != invalid_id
    for record in safe_records
)

# Restore dataset variables used by subsequent cells.
train_records = safe_records

train_dataset = AmharicASRDataset(
    train_records,
    AUDIO_DIR,
    processor
)

assert len(train_dataset) == 5083

# Report available checkpoints.
print("DATASET RECOVERY")
print("-" * 45)
print("Recordings:", len(train_dataset))
print("Unique speakers:", len(set(
    record["speaker_id"] for record in train_records
)))
print("Invalid recording excluded: YES")

print("\nSAVED TRAINING CHECKPOINTS")

for step in [10, 50, 200, 350, 500]:
    path = CHECKPOINT_DIR / f"step_{step:06d}"
    print(
        f"Step {step}:",
        "AVAILABLE" if (path / "COMPLETE").exists()
        else "NOT FOUND"
    )

print("\nCurrent in-memory training step:",
      globals().get("successful_updates", "NOT LOADED"))

print("\nCTC-SAFE DATASET RESTORED")
